# Sequential YOLO Training on L4 GPU - Fire/Smoke Detection

## Analysis Results:

### 1. GPU Optimization Status ✅
- **All models are GPU-optimized**: Nano/Tiny versions are perfect for L4
- **AMP enabled**: `amp=device.startswith("cuda")` provides 2x speedup
- **Batch size 32**: Maximizes L4's 22GB VRAM
- **Mixed precision**: Reduces memory usage, increases speed

### 2. Model Verification ✅
- **YOLOv8n**: 3.2M params, fastest training
- **YOLOv9t**: 2.0M params, excellent accuracy/speed balance
- **YOLO11n**: 2.6M params, latest architecture
- **YOLO12n**: ~2.5M params, newest model

### 3. Project Suitability ✅
- **Fire/Smoke detection**: YOLO is perfect for real-time detection
- **CNN-based**: All YOLO models use advanced CNN backbones
- **Production ready**: Models can be deployed to edge devices

This notebook will train all 4 models sequentially and compare results.

In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


In [ ]:
#@title Setup Environment and Clone Repo
import os, shutil
import torch

project_dir = "/content/jetsonorin"
repo_url = "https://github.com/e-seet/jetsonorin.git"
branch_name = "colab"

os.chdir("/content")
if os.path.exists(project_dir):
    shutil.rmtree(project_dir)

!git clone -b {branch_name} "$repo_url" "$project_dir"
os.chdir(project_dir)
print("Working directory:", os.getcwd())

In [ ]:
#@title Install GPU-Enabled Dependencies
%%bash
pip install -U pip
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
pip install "ultralytics<9" kagglehub opencv-python pyyaml polars seaborn matplotlib pandas requests tqdm psutil

In [ ]:
#@title Verify L4 GPU and Model Compatibility
import torch
from ultralytics import YOLO

print("=== L4 GPU Verification ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    
    # Test model loading
    models = {
        "YOLOv8n": "yolov8n.pt",
        "YOLOv9t": "yolov9t.pt", 
        "YOLO11n": "yolo11n.pt",
        "YOLO12n": "yolo12n.pt"
    }
    
    print("\n=== Model Compatibility Check ===")
    for name, model_file in models.items():
        try:
            model = YOLO(model_file)
            params = sum(p.numel() for p in model.model.parameters())
            print(f"✅ {name}: {params:,} parameters - GPU compatible")
            del model
            torch.cuda.empty_cache()
        except Exception as e:
            print(f"❌ {name}: Error - {e}")
else:
    print("❌ No GPU detected!")

In [ ]:
#@title Setup Kaggle API and Download Dataset
import os

# Set Kaggle token
KAGGLE_API_TOKEN = "e5c444b8b13963507dd32b1746492fe7"
os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN

# Download dataset
%%bash
set -euo pipefail
cd /content/jetsonorin
python download_dataset.py
python preprocess_data.py --data datasets/fire_smoke/data.yaml

In [ ]:
#@title Fix Data Paths for Colab
from pathlib import Path
import yaml

data_yaml_path = Path("datasets/fire_smoke/data.yaml")
with data_yaml_path.open() as f:
    data_cfg = yaml.safe_load(f)

data_cfg["path"] = "datasets/fire_smoke"
data_cfg["train"] = "data/train/images"
data_cfg["val"] = "data/val/images"
data_cfg["test"] = "data/test/images"

with data_yaml_path.open("w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print("✅ Data paths configured for Colab")

## Sequential Training - All 4 Models

**Training Order (Fastest to Slowest):**
1. YOLOv9t (2.0M params) - Fastest, good baseline
2. YOLOv8n (3.2M params) - Standard YOLOv8
3. YOLO11n (2.6M params) - Latest architecture
4. YOLO12n (~2.5M params) - Newest model

**L4 Optimizations Applied:**
- Batch size: 32 (max for 22GB VRAM)
- Mixed precision: Enabled
- Workers: 8 (parallel loading)
- Image size: 512 (good balance)

In [ ]:
#@title Sequential Training - All Models
import torch
import time
import pandas as pd
from pathlib import Path

# L4 Optimized parameters
BATCH_SIZE = 32
IMG_SIZE = 512
EPOCHS = 50  # Reduced for comparison, increase to 80+ for production
WORKERS = 8
DEVICE = "0"  # L4 GPU

# Models in order of training speed
models_to_train = [
    ("yolov9", "YOLOv9t - Tiny (Fastest)"),
    ("yolov8", "YOLOv8n - Nano"),
    ("yolo11", "YOLO11n - Latest"),
    ("yolo12", "YOLO12n - Newest")
]

results = []
total_start_time = time.time()

print("🚀 Starting Sequential YOLO Training on L4 GPU")
print(f"📊 Batch Size: {BATCH_SIZE}, Image Size: {IMG_SIZE}x{IMG_SIZE}, Epochs: {EPOCHS}")
print("=" * 60)

for i, (model_key, model_desc) in enumerate(models_to_train, 1):
    print(f"\n🔥 Training {i}/4: {model_desc}")
    print("-" * 40)
    
    start_time = time.time()
    
    # Build training command
    cmd = f"python train_yolo.py --models {model_key} " \
          f"--data datasets/fire_smoke/data.yaml " \
          f"--epochs {EPOCHS} --batch {BATCH_SIZE} " \
          f"--imgsz {IMG_SIZE} --workers {WORKERS} " \
          f"--device {DEVICE} --cache False"
    
    print(f"📝 Command: {cmd}")
    
    # Execute training
    !{cmd}
    
    end_time = time.time()
    training_time = end_time - start_time
    
    # Check if training completed successfully
    weights_path = Path(f"fire_detection_runs_v2/{model_key}_fire/weights/best.pt")
    if weights_path.exists():
        file_size = weights_path.stat().st_size / 1024**2  # MB
        results.append({
            'Model': model_desc,
            'Key': model_key,
            'Training Time (min)': round(training_time / 60, 1),
            'Model Size (MB)': round(file_size, 1),
            'Status': '✅ Success'
        })
        print(f"✅ Completed in {training_time/60:.1f} minutes - Model: {file_size:.1f} MB")
    else:
        results.append({
            'Model': model_desc,
            'Key': model_key,
            'Training Time (min)': round(training_time / 60, 1),
            'Model Size (MB)': 'N/A',
            'Status': '❌ Failed'
        })
        print(f"❌ Training failed after {training_time/60:.1f} minutes")
    
    # Clear GPU memory before next model
    torch.cuda.empty_cache()

total_time = time.time() - total_start_time

print("\n" + "=" * 60)
print("📊 TRAINING SUMMARY")
print("=" * 60)

# Display results
df = pd.DataFrame(results)
print(df.to_string(index=False))

print(f"\n⏱️  Total Training Time: {total_time/60:.1f} minutes")
print(f"🏆 Successful Models: {sum(1 for r in results if '✅' in r['Status'])}/4")

## Model Benchmarking and Comparison

Compare the trained models on validation set to find the best performer.

In [ ]:
#@title Benchmark All Trained Models
import torch
from ultralytics import YOLO
import pandas as pd
from pathlib import Path
import time

def benchmark_model(model_path, model_name):
    """Benchmark a trained model on validation set"""
    try:
        model = YOLO(model_path)
        
        # Measure inference speed
        start_time = time.time()
        results = model.val(
            data="datasets/fire_smoke/data.yaml",
            imgsz=512,
            batch=32,
            device="0",
            plots=False,
            save_json=False
        )
        inference_time = time.time() - start_time
        
        # Extract metrics
        metrics = results.results_dict
        
        return {
            'Model': model_name,
            'mAP50': round(metrics['metrics/mAP50(B)'], 3),
            'mAP50-95': round(metrics['metrics/mAP50-95(B)'], 3),
            'Precision': round(metrics['metrics/precision(B)'], 3),
            'Recall': round(metrics['metrics/recall(B)'], 3),
            'Inference Time (s)': round(inference_time, 1),
            'FPS': round(len(results.dataset) / inference_time, 1)
        }
    except Exception as e:
        print(f"Error benchmarking {model_name}: {e}")
        return {
            'Model': model_name,
            'mAP50': 'N/A',
            'mAP50-95': 'N/A', 
            'Precision': 'N/A',
            'Recall': 'N/A',
            'Inference Time (s)': 'N/A',
            'FPS': 'N/A'
        }

print("🔍 Benchmarking all trained models...")
print("=" * 50)

benchmark_results = []
models_to_test = [
    ("fire_detection_runs_v2/yolov9_fire/weights/best.pt", "YOLOv9t"),
    ("fire_detection_runs_v2/yolov8_fire/weights/best.pt", "YOLOv8n"),
    ("fire_detection_runs_v2/yolo11_fire/weights/best.pt", "YOLO11n"),
    ("fire_detection_runs_v2/yolo12_fire/weights/best.pt", "YOLO12n")
]

for model_path, model_name in models_to_test:
    if Path(model_path).exists():
        print(f"📊 Testing {model_name}...")
        result = benchmark_model(model_path, model_name)
        benchmark_results.append(result)
    else:
        print(f"❌ Model not found: {model_path}")
        benchmark_results.append({
            'Model': model_name,
            'mAP50': 'N/A',
            'mAP50-95': 'N/A',
            'Precision': 'N/A', 
            'Recall': 'N/A',
            'Inference Time (s)': 'N/A',
            'FPS': 'N/A'
        })

# Display benchmark results
df_benchmark = pd.DataFrame(benchmark_results)
print("\n📈 BENCHMARK RESULTS")
print("=" * 50)
print(df_benchmark.to_string(index=False))

# Find best model by mAP50
valid_results = [r for r in benchmark_results if r['mAP50'] != 'N/A']
if valid_results:
    best_model = max(valid_results, key=lambda x: x['mAP50'])
    print(f"\n🏆 BEST MODEL: {best_model['Model']} (mAP50: {best_model['mAP50']})")
    print(f"⚡ Fastest: {max(valid_results, key=lambda x: x['FPS'])['Model']} ({max(valid_results, key=lambda x: x['FPS'])['FPS']} FPS)")

## Save Results and Best Model

Save the best performing model and training summary to Google Drive.

In [ ]:
#@title Save Best Model and Results to Google Drive
import shutil
from pathlib import Path
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create results directory
results_dir = Path("/content/drive/MyDrive/yolo_fire_detection_results")
results_dir.mkdir(parents=True, exist_ok=True)

# Save training summary
if 'results' in locals():
    df_training = pd.DataFrame(results)
    df_training.to_csv(results_dir / "training_summary.csv", index=False)
    print("💾 Training summary saved to Google Drive")

# Save benchmark results
if 'benchmark_results' in locals():
    df_benchmark = pd.DataFrame(benchmark_results)
    df_benchmark.to_csv(results_dir / "benchmark_results.csv", index=False)
    print("📊 Benchmark results saved to Google Drive")

# Find and save best model
if 'benchmark_results' in locals():
    valid_results = [r for r in benchmark_results if r['mAP50'] != 'N/A']
    if valid_results:
        best_model = max(valid_results, key=lambda x: x['mAP50'])
        model_name = best_model['Model']
        
        # Map model names to file paths
        model_paths = {
            "YOLOv9t": "fire_detection_runs_v2/yolov9_fire/weights/best.pt",
            "YOLOv8n": "fire_detection_runs_v2/yolov8_fire/weights/best.pt",
            "YOLO11n": "fire_detection_runs_v2/yolo11_fire/weights/best.pt",
            "YOLO12n": "fire_detection_runs_v2/yolo12_fire/weights/best.pt"
        }
        
        if model_name in model_paths:
            source_path = Path(model_paths[model_name])
            if source_path.exists():
                dest_path = results_dir / f"best_model_{model_name.lower()}.pt"
                shutil.copy2(source_path, dest_path)
                print(f"🏆 Best model ({model_name}) saved to Google Drive")
                print(f"📍 Location: {dest_path}")
                print(f"📏 Size: {source_path.stat().st_size / 1024**2:.1f} MB")

print(f"\n📁 All results saved to: {results_dir}")
print("✅ Ready for deployment to edge devices!")